In [1]:
!pip install selenium pandas webdriver-manager beautifulsoup4

In [2]:
import sys
!{sys.executable} -m pip install selenium pandas webdriver-manager beautifulsoup4 lxml

In [3]:
import time
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


def scrape_canada_respiratory_fixed():
    url = "https://health-infobase.canada.ca/respiratory-virus-surveillance/explore.html#visualize"
    
    options = Options()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    all_data = []

    try:
        driver.get(url)
        wait = WebDriverWait(driver, 30)

        print("🔄 Page Access...")
        
        # "WEEK" dropdown box searching
        dropdown_element = wait.until(EC.presence_of_element_located((By.ID, "table1WeekDD")))
        print("✅ Dropdown box successfully found!")

        # Scrape all the weeks available
        select = Select(dropdown_element)
        weeks = [opt.text for opt in select.options if opt.get_attribute("value")]
        print(f"📊 Total {len(weeks)} weeks data found!")

        for week_text in weeks:
            print(f"🚀 Processing : {week_text}")
            
            # Select weeks
            dropdown = Select(driver.find_element(By.ID, "table1WeekDD"))
            dropdown.select_by_visible_text(week_text)
            
            time.sleep(4) 
            
            # 4. BeautifulSoup parsing
            soup = BeautifulSoup(driver.page_source, "html.parser")
            tables = soup.find_all("table")
            
            found_canada = False
            for table in tables:
                # Searching "CANADA" within the Table 1
                if "CANADA" in table.get_text().upper():
                    df_list = pd.read_html(StringIO(str(table)))
                    if not df_list: continue
                    df = df_list[0]
                    
                    # List column names
                    if isinstance(df.columns, pd.MultiIndex):
                        df.columns = [str(c[-1]).strip() for c in df.columns]
                    else:
                        df.columns = [str(c).strip() for c in df.columns]

                    # Extract row where it starts with CANADA
                    is_canada = df.iloc[:, 0].astype(str).str.upper().str.contains("CANADA", na=False)
                    canada_row = df[is_canada].tail(1).copy()
                    
                    if not canada_row.empty:
                        # Add year and week information
                        canada_row.insert(0, "Selected_Week", week_text)
                        all_data.append(canada_row)
                        found_canada = True
                        print(f"   [Success] CANADA row extracted")
                        break
            
            if not found_canada:
                print(f"   ⚠️ {week_text}: CANADA row not found")

    except Exception as e:
        print(f"\n❌ Error: {e}")
    finally:
        driver.quit()

    # Convert to CSV
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        final_df = final_df.drop_duplicates(subset=["Selected_Week"])
        output_filename = "virus_last21weeks.csv"
        final_df.to_csv(output_filename, index=False, encoding="utf-8-sig")
        print(f"\n✨ Success! File saved: {output_filename} (Total {len(final_df)}rows)")
    else:
        print("\n Data is not scraped.")


if __name__ == "__main__":
    scrape_canada_respiratory_fixed()

🔄 Page Access...
✅ Dropdown box successfully found!
📊 Total 21 weeks data found!
🚀 Processing : week 2 (week ending January 17, 2026)
   [Success] CANADA row extracted
🚀 Processing : week 1 (week ending January 10, 2026)
   [Success] CANADA row extracted
🚀 Processing : week 53 (week ending January 3, 2026)
   [Success] CANADA row extracted
🚀 Processing : week 52 (week ending December 27, 2025)
   [Success] CANADA row extracted
🚀 Processing : week 51 (week ending December 20, 2025)
   [Success] CANADA row extracted
🚀 Processing : week 50 (week ending December 13, 2025)
   [Success] CANADA row extracted
🚀 Processing : week 49 (week ending December 6, 2025)
   [Success] CANADA row extracted
🚀 Processing : week 48 (week ending November 29, 2025)
   [Success] CANADA row extracted
🚀 Processing : week 47 (week ending November 22, 2025)
   [Success] CANADA row extracted
🚀 Processing : week 46 (week ending November 15, 2025)
   [Success] CANADA row extracted
🚀 Processing : week 45 (week ending 

In [4]:
import pandas as pd
import re


def process_respiratory_data_pipeline(file_path):
    """
    Pipeline to load respiratory virus data and 
    generate integrated indicators for supply chain pressure analysis.
    """
    # 1. Load Data
    df = pd.read_csv(file_path)
    
    # 2. Drop unnecessary geographical columns (as we use Canada total)
    if 'Reporting laboratory' in df.columns:
        df = df.drop("Reporting laboratory", axis=1)
    
    # 3. Create Integrated Indicators (COVID + Flu + RSV)
    # We sum these values to understand the total burden on the supply chain.
    df['Total_Test_Number'] = df['SARS-CoV-2 tests'] + df['Influenza tests'] + df['RSV tests']
    df['Total_Detection'] = df['SARS-CoV-2 detections'] + df['Total influenza detections'] + df['RSV detections']
    
    # 4. Calculate Aggregate Positive Rate
    # fillna(0) prevents errors in case of zero tests in a specific week
    df['Positive_Rate'] = (df['Total_Detection'] / df['Total_Test_Number']).fillna(0)
    
    # 5. Drop Individual Disease Columns
    # We focus on overall supply pressure, so we remove individual metrics 
    # to reduce model complexity.
    drop_cols = [
        "SARS-CoV-2 tests", "SARS-CoV-2 detections", 
        "Influenza tests", "Total influenza detections", 
        "RSV tests", "RSV detections"
    ]
    df = df.drop(drop_cols, axis=1)
    
    # 6. Parse 'Selected_Week' Column (Using Regex)
    # Example: "week 1 (week ending January 10, 2026)" -> [1, "January 10", 2026]
    pattern = r"week (\d+) \(week ending (.*?), (\d{4})\)"
    df[['Week_Number', 'Ending_Date', 'Year']] = df['Selected_Week'].str.extract(pattern)
    
    # 7. Convert Data Types and Drop Original Column
    df['Week_Number'] = df['Week_Number'].astype(int)
    df['Year'] = df['Year'].astype(int)
    df = df.drop("Selected_Week", axis=1)
    
    # 8. Reorder Columns for Clarity
    cols_order = ['Year', 'Week_Number', 'Ending_Date', 'Total_Test_Number', 'Total_Detection', 'Positive_Rate']
    df = df[cols_order]
    
    # 9. Sort Chronologically (Past -> Present)
    # Essential for training time-series models (e.g., LSTM, Prophet)
    df = df.sort_values(['Year', 'Week_Number']).reset_index(drop=True)
    
    return df



input_file = "virus_last21weeks.csv"
final_df = process_respiratory_data_pipeline(input_file)

print("✅ Preprocessing Complete. Top 5 rows of final dataframe:")
print(final_df.head())

# Save as CSV for Machine Learning Training
final_df.to_csv("Processed_Virus_Detection.csv", index=False)

✅ Preprocessing Complete. Top 5 rows of final dataframe:
   Year  Week_Number   Ending_Date  Total_Test_Number  Total_Detection  \
0  2025           35     August 30            46149.0           1442.0   
1  2025           36   September 6            48796.0           1763.0   
2  2025           37  September 13            53601.0           2137.0   
3  2025           38  September 20            61288.0           2626.0   
4  2025           39  September 27            66474.0           3119.0   

   Positive_Rate  
0       0.031247  
1       0.036130  
2       0.039869  
3       0.042847  
4       0.046921  
